### 1. Load data

In [1]:
import polars as pl

df_metadata = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/KuaiRand-1k-item_metadata_temp-parquet/kaggle/working/item_metadata_temp.parquet")
df_statistics = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/KuaiRand-1k-item_statistics-parquet/kaggle/working/video_ranking_statistics.parquet")
df_users = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/KuaiRand-1k-user_table-parquet/kaggle/working/user_table.parquet")
df_items = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/KuaiRand-1k-item_table-parquet/kaggle/working/item_table.parquet")

print("=========================================")
print("Metadata")
print("==========================================")
print(df_metadata)

print("\n==========================================")
print("User features")
print("==========================================")
print(df_users)

print("\n==========================================")
print("Item features")
print("==========================================")
print(df_items)

print("\n==========================================")
print("Item statistics")
print("==========================================")
print(df_statistics)

Metadata
shape: (4_371_868, 12)
┌──────────┬───────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬─────┐
│ video_id ┆ author_id ┆ first_level ┆ second_lev ┆ … ┆ music_type ┆ upload_tim ┆ video_type ┆ tag │
│ ---      ┆ ---       ┆ _category_n ┆ el_categor ┆   ┆ ---        ┆ estamp     ┆ ---        ┆ --- │
│ i32      ┆ i32       ┆ ame         ┆ y_name     ┆   ┆ f64        ┆ ---        ┆ str        ┆ str │
│          ┆           ┆ ---         ┆ ---        ┆   ┆            ┆ i64        ┆            ┆     │
│          ┆           ┆ str         ┆ str        ┆   ┆            ┆            ┆            ┆     │
╞══════════╪═══════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪═════╡
│ 304621   ┆ 8339677   ┆ 游戏        ┆ 吃鸡游戏   ┆ … ┆ 9.0        ┆ 1651795200 ┆ NORMAL     ┆ 3   │
│ 304137   ┆ 6680528   ┆ 财经        ┆ UNKNOWN    ┆ … ┆ 9.0        ┆ 1649808000 ┆ NORMAL     ┆ 13  │
│ 305629   ┆ 259258    ┆ 游戏        ┆ 动作角色扮 ┆ … ┆ 9.0        ┆ 16502

### 2. Model definition and loss function

In [2]:
from __future__ import annotations

from collections.abc import Sequence

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence

from dataclasses import asdict, dataclass
from math import isfinite

@dataclass
class ModelConfig:
    output_dim: int = 64
    history_video_dim: int = 64
    gru_hidden_dim: int = 64 #hidden state
    gru_layers: int = 1 #the number of stacked GRU layers.

    #Categorical embeddings
    item_video_dim: int = 64 #video id
    video_type_dim: int = 8 #video_type
    music_type_dim: int = 8 #music_type
    tag_dim: int = 16
    
    nlp_input_dim: int = 512 #nlp_vector
    nlp_projected_dim: int = 128 #projected nlp_Vector

    user_mlp_hidden: tuple[int, ...] = (256, 128) #150 -> 256 -> 128 -> 64
    item_mlp_hidden: tuple[int, ...] = (256, 128) #224 -> 256 -> 128 -> 64
    dropout: float = 0.10 #turns off neurons during neural network TRAINING to prevent overfitting

def make_mlp(input_dim: int, hidden_dims: Sequence[int], output_dim: int, dropout: float) -> nn.Sequential:
    layers: list[nn.Module] = []
    current = input_dim
    for hidden in hidden_dims:
        layers.extend([
            nn.Linear(current, hidden), #224 -> 256
            nn.LayerNorm(hidden), #Keeps the mean and variance of layer inputs consistent, smoothing the optimization landscape and stopping values from growing or shrinking exponentially through deep layers
            nn.GELU(), #smooth non-linear relationship
            nn.Dropout(dropout), #reduce overfitting
        ])
        current = hidden
    layers.append(nn.Linear(current, output_dim)) #128 -> 64
    return nn.Sequential(*layers)


class UserTower(nn.Module):
    def __init__(
        self,
        video_vocab_size: int, #total number of video_id
        numeric_dim: int, # 
        categorical_cardinalities: Sequence[int], #numbers of 18 onehot feats
        categorical_dims: Sequence[int], #dimensions of 18 onehot feats
        config: ModelConfig,
    ):
        super().__init__()
        if len(categorical_cardinalities) != len(categorical_dims):
            raise ValueError("User categorical cardinalities and dimensions differ")
        self.config = config
        self.numeric_dim = numeric_dim
        self.history_embedding = nn.Embedding(
            video_vocab_size, config.history_video_dim, padding_idx=0
        )
        self.gru = nn.GRU(     #(B,20,64) -> GRU
            input_size=config.history_video_dim,
            hidden_size=config.gru_hidden_dim,
            num_layers=config.gru_layers,
            batch_first=True,
        )
        self.categorical_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dimension, padding_idx=0)
            for cardinality, dimension in zip(categorical_cardinalities, categorical_dims)
        ])
        input_dim = config.gru_hidden_dim + numeric_dim + sum(categorical_dims)
        self.mlp = make_mlp(
            input_dim, config.user_mlp_hidden, config.output_dim, config.dropout
        )

    def forward(
        self,
        history_video_indices: torch.Tensor, #(B, 20)
        history_lengths: torch.Tensor, #(B,)
        numeric_features: torch.Tensor, #(B,2): is is_lowactive_period, user_active_degree
        categorical_features: torch.Tensor, #(B, 18)
    ) -> torch.Tensor:
        if history_video_indices.ndim != 2:
            raise ValueError("history_video_indices must be shape (B, L)")
        batch_size, padded_length = history_video_indices.shape
        if history_lengths.shape != (batch_size,):
            raise ValueError("Invalid history shapes")
        if history_video_indices.dtype not in (torch.int32, torch.int64):
            raise ValueError("history_video_indices must contain integer indices")
        if history_lengths.dtype not in (torch.int32, torch.int64):
            raise ValueError("history_lengths must contain integers")
        if ((history_lengths < 0) | (history_lengths > padded_length)).any():
            raise ValueError("history_lengths must be between 0 and the padded history length")
        if numeric_features.shape != (batch_size, self.numeric_dim):
            raise ValueError(f"numeric_features must be shape (B, {self.numeric_dim})")
        if not numeric_features.is_floating_point():
            raise ValueError("numeric_features must be floating point after preprocessing")
        if categorical_features.dtype not in (torch.int32, torch.int64):
            raise ValueError("User categorical features must contain integer indices")
        if categorical_features.shape != (batch_size, len(self.categorical_embeddings)):
            raise ValueError("Invalid user categorical feature shape")

        #Some users might have no positive historical interactions.
        history_state = self.history_embedding.weight.new_zeros(
            (batch_size, self.config.gru_hidden_dim)
        )
        nonempty = history_lengths > 0 #masking list of who has non-empty history
        if nonempty.any():
            embedded = self.history_embedding(history_video_indices[nonempty])
            packed = pack_padded_sequence(
                embedded,
                history_lengths[nonempty].detach().cpu(),
                batch_first=True,
                enforce_sorted=False,
            )
            _, hidden = self.gru(packed)
            history_state[nonempty] = hidden[-1]

        categorical_parts = [
            embedding(categorical_features[:, i])
            for i, embedding in enumerate(self.categorical_embeddings)
        ]
        combined = torch.cat(
            [history_state, numeric_features, *categorical_parts], dim=-1
        )
        return F.normalize(self.mlp(combined), p=2, dim=-1)


class ItemTower(nn.Module):
    def __init__(
        self,
        video_vocab_size: int, #total numbers of video_id
        categorical_cardinalities: Sequence[int], #numbers of video_type, music_type, tag
        config: ModelConfig,
    ):
        super().__init__()
        if len(categorical_cardinalities) != 3:
            raise ValueError("Expected categorical fields: video_type, music_type, tag")
        self.nlp_input_dim = config.nlp_input_dim
        self.video_embedding = nn.Embedding(
            video_vocab_size, config.item_video_dim, padding_idx=0
        )
        dims = [config.video_type_dim, config.music_type_dim, config.tag_dim]
        self.categorical_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dimension, padding_idx=0)
            for cardinality, dimension in zip(categorical_cardinalities, dims)
        ])
        self.nlp_projection = nn.Sequential(
            nn.Linear(config.nlp_input_dim, config.nlp_projected_dim),
            nn.LayerNorm(config.nlp_projected_dim),
            nn.GELU(),
        )
        concat_dim = config.item_video_dim + sum(dims) + config.nlp_projected_dim
        if concat_dim != 224:
            raise ValueError(f"Item concatenation must be 224-D; got {concat_dim}")
        self.mlp = make_mlp(
            concat_dim, config.item_mlp_hidden, config.output_dim, config.dropout
        )

    def forward(
        self,
        video_indices: torch.Tensor, #(B,)
        categorical_features: torch.Tensor, #(B, 3): 3 item categorical features
        nlp_vectors: torch.Tensor,
    ) -> torch.Tensor:
        if video_indices.ndim != 1:
            raise ValueError("video_indices must be shape (B,)")
        if video_indices.dtype not in (torch.int32, torch.int64):
            raise ValueError("video_indices must contain integer indices")
        if categorical_features.shape != (video_indices.shape[0], 3):
            raise ValueError("Invalid item categorical feature shape")
        if categorical_features.dtype not in (torch.int32, torch.int64):
            raise ValueError("Item categorical features must contain integer indices")
        if nlp_vectors.shape != (video_indices.shape[0], self.nlp_input_dim):
            raise ValueError(f"nlp_vectors must be shape (B, {self.nlp_input_dim})")
        if not nlp_vectors.is_floating_point():
            raise ValueError("nlp_vectors must be floating point")
        parts = [self.video_embedding(video_indices)]
        parts.extend(
            embedding(categorical_features[:, i])
            for i, embedding in enumerate(self.categorical_embeddings)
        )
        parts.append(self.nlp_projection(nlp_vectors))
        return F.normalize(self.mlp(torch.cat(parts, dim=-1)), p=2, dim=-1)


class TwoTowerModel(nn.Module):
    def __init__(self, user_tower: UserTower, item_tower: ItemTower):
        super().__init__()
        self.user_tower = user_tower
        self.item_tower = item_tower

    def encode_user(self, batch: dict[str, torch.Tensor]) -> torch.Tensor:
        return self.user_tower(
            batch["history_video_indices"],
            batch["history_lengths"],
            batch["user_numeric"],
            batch["user_categorical"],
        )

    def encode_item(self, batch: dict[str, torch.Tensor]) -> torch.Tensor:
        return self.item_tower(
            batch["target_video_indices"],
            batch["item_categorical"],
            batch["item_nlp"],
        )

    def forward(self, batch: dict[str, torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
        return self.encode_user(batch), self.encode_item(batch)


def masked_inbatch_info_nce(
    user_embeddings: torch.Tensor, #(B,64)
    item_embeddings: torch.Tensor, #(B,64)
    target_video_indices: torch.Tensor, #(B, )
    temperature: float,
    target_log_q: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """InfoNCE with duplicate-positive columns removed from other rows."""
    if user_embeddings.ndim != 2 or item_embeddings.ndim != 2:
        raise ValueError("User and item embeddings must be shape (B, D)")
    if user_embeddings.shape != item_embeddings.shape:
        raise ValueError("User and item embedding shapes must match")
    if user_embeddings.shape[0] == 0 or user_embeddings.shape[1] == 0:
        raise ValueError("Embeddings must have a nonempty batch and feature dimension")
    if user_embeddings.device != item_embeddings.device:
        raise ValueError("User and item embeddings must be on the same device")
    if not user_embeddings.is_floating_point() or not item_embeddings.is_floating_point():
        raise ValueError("User and item embeddings must be floating point")
    batch_size = user_embeddings.shape[0]
    if target_video_indices.shape != (batch_size,):
        raise ValueError("target_video_indices must be shape (B,)")
    if target_video_indices.dtype not in (torch.int32, torch.int64):
        raise ValueError("target_video_indices must contain integer indices")
    if target_video_indices.device != user_embeddings.device:
        raise ValueError("target_video_indices must be on the embeddings device")
    if not isfinite(temperature) or temperature <= 0:
        raise ValueError("temperature must be finite and positive")

    with torch.autocast(device_type=user_embeddings.device.type, enabled=False):
        logits = user_embeddings.float() @ item_embeddings.float().T / temperature
        #LogQ Correction -> Popularity Bias
        if target_log_q is not None:
            if target_log_q.shape != (batch_size,):
                raise ValueError("target_log_q must be shape (B,)")
            target_log_q = target_log_q.to(device=logits.device, dtype=torch.float32)
            if not torch.isfinite(target_log_q).all():
                raise ValueError("target_log_q must contain only finite values")
            logits = logits - target_log_q.unsqueeze(0)

        #False negative
        same_target = target_video_indices[:, None].eq(target_video_indices[None, :]) #Same target at i and j?
        diagonal = torch.eye(batch_size, dtype=torch.bool, device=logits.device)
        invalid_negative = same_target & ~diagonal #Find false negatives not in diagonal
        logits = logits.masked_fill(invalid_negative, float("-inf")) #do not care duplicates
        labels = torch.arange(batch_size, device=logits.device)
        loss = F.cross_entropy(logits, labels)
        return loss, logits

### 3. Data loader and precprocessing

In [17]:
from __future__ import annotations

import gc
import json
import math
import os
import shutil
import tempfile
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch.utils.data import DataLoader, Dataset

USER_PATH = Path('/kaggle/input/datasets/nguyenngocanhle/two-tower-data/KuaiRand-1k-user_table-parquet/kaggle/working/user_table.parquet')
ITEM_PATH = Path('/kaggle/input/datasets/nguyenngocanhle/two-tower-data/KuaiRand-1k-item_table-parquet/kaggle/working/item_table.parquet')
OUTPUT_DIR = Path('/kaggle/working/kuairand_retrieval_ready_v2')

SIGNALS = (
    'is_hate', 'is_click', 'long_view', 'is_like',
    'is_follow', 'is_comment', 'is_forward', 'is_profile_enter',
)
USER_CATEGORICAL_COLUMNS = tuple(f'onehot_feat{i}' for i in range(18))
USER_CATEGORICAL_DIMS = (1, 4, 8, 16, 4, 8, 1, 8, 16, 4, 4, 4, 1, 1, 1, 1, 1, 1)
ITEM_CATEGORICAL_COLUMNS = ('video_type', 'music_type', 'tag')
ACTIVE_DEGREE_TO_CODE = {
    'high_active': 0,
    'full_active': 1,
    'middle_active': 2,
    'low_active': 3,
    'single_low_active': 4,
    '2_14_day_new': 5,
    '30day_retention': 6,
    'UNKNOWN': 7,
}
PAD_INDEX = 0
UNK_INDEX = 1
MISSING_UPLOAD_MS = np.iinfo(np.int64).min
FORMAT_VERSION = 2

assert sum(USER_CATEGORICAL_DIMS) == 84
assert 64 + 2 + sum(USER_CATEGORICAL_DIMS) == 150


@dataclass(frozen=True)
class PreprocessConfig:
    exposure_limit: int = 150
    history_length: int = 20
    train_fraction: float = 0.80
    nlp_dimension: int = 512
    item_batch_size: int = 8192
    upload_seconds_to_ms: int = 1000
    log_q_floor: float = 1e-12

    def validate(self) -> None:
        if self.exposure_limit <= self.history_length or self.history_length < 1:
            raise ValueError('Require exposure_limit > history_length >= 1')
        if not 0.0 < self.train_fraction < 1.0:
            raise ValueError('train_fraction must be between 0 and 1')
        if self.nlp_dimension < 1 or self.item_batch_size < 1:
            raise ValueError('nlp_dimension and item_batch_size must be positive')
        if self.upload_seconds_to_ms < 1:
            raise ValueError('upload_seconds_to_ms must be positive')
        if not 0.0 < self.log_q_floor < 1.0:
            raise ValueError('log_q_floor must be between 0 and 1')

## -----------------------------------
## ENCODING AND VALIDATING HELPER FUNCTIONS
## -----------------------------------

def canonical_category(value: object) -> str | None:
    if value is None:
        return None
    if isinstance(value, (float, np.floating)) and not math.isfinite(float(value)):
        return None
    text = str(value).strip()
    return text if text else None


def fit_category_vocab(values) -> dict[str, int]:
    tokens = sorted({token for value in values if (token := canonical_category(value)) is not None})
    return {token: index + 2 for index, token in enumerate(tokens)}


def encode_category_values(values, vocab: dict[str, int]) -> np.ndarray:
    return np.asarray(
        [vocab.get(canonical_category(value), UNK_INDEX) for value in values],
        dtype=np.int32,
    )


def encode_video_ids(original_ids, sorted_catalog_ids: np.ndarray) -> np.ndarray:
    original_ids = np.asarray(original_ids, dtype=np.int64)
    if not len(original_ids):
        return np.empty(0, dtype=np.int32)
    positions = np.searchsorted(sorted_catalog_ids, original_ids)
    clipped = np.minimum(positions, len(sorted_catalog_ids) - 1)
    known = (positions < len(sorted_catalog_ids)) & (sorted_catalog_ids[clipped] == original_ids)
    return np.where(known, positions + 2, UNK_INDEX).astype(np.int32)


def fixed_vectors_to_numpy(column: pa.Array, expected_width: int) -> np.ndarray:
    if isinstance(column, pa.ChunkedArray):
        column = column.combine_chunks()
    if column.null_count:
        raise ValueError('nlp_vector contains null rows')
    if pa.types.is_fixed_size_list(column.type):
        if column.type.list_size != expected_width:
            raise ValueError(f'Expected NLP width {expected_width}; got {column.type.list_size}')
        values = column.values.slice(column.offset * expected_width, len(column) * expected_width)
        if values.null_count:
            raise ValueError('nlp_vector contains null elements')
        result = np.asarray(values.to_numpy(zero_copy_only=False), dtype=np.float32)
        result = result.reshape(len(column), expected_width)
    else:
        try:
            result = np.asarray(column.to_pylist(), dtype=np.float32)
        except (TypeError, ValueError) as error:
            raise ValueError('nlp_vector must contain equal-length numeric vectors') from error
    if result.shape != (len(column), expected_width) or not np.isfinite(result).all():
        raise ValueError(f'Expected finite NLP array with shape (N, {expected_width})')
    return result


def map_active_degree(value: object) -> float:
    if value is None:
        return float(ACTIVE_DEGREE_TO_CODE['UNKNOWN'])
    if isinstance(value, (int, np.integer)) and int(value) in (0, 1, 2, 3):
        return float(value)
    text = str(value).strip()
    if text in ACTIVE_DEGREE_TO_CODE:
        return float(ACTIVE_DEGREE_TO_CODE[text])
    if text in {'0', '1', '2', '3', '4', '5', '6'}:
        return float(int(text))
    raise ValueError(f'Unexpected user_active_degree value: {value!r}')


def is_positive_feedback(feedback: dict) -> bool:
    if not isinstance(feedback, dict):
        raise ValueError('Each history_feedback entry must be a struct/dict')
    missing = set(SIGNALS) - set(feedback)
    if missing:
        raise ValueError(f'history_feedback is missing signals: {sorted(missing)}')
    values = {}
    for signal in SIGNALS:
        value = 0 if feedback[signal] is None else feedback[signal]
        if value not in (0, 1):
            raise ValueError(f'{signal} must be binary; received {value!r}')
        values[signal] = int(value)
    return values['is_hate'] == 0 and any(values[name] == 1 for name in SIGNALS[1:])


def validate_source_schema(user_frame: pl.DataFrame, item_path: str | Path) -> None:
    user_required = {
        'user_id', 'history_sequence', 'history_time_ms', 'history_feedback',
        'is_lowactive_period', 'user_active_degree', *USER_CATEGORICAL_COLUMNS,
    }
    item_required = {
        'video_id', 'upload_timestamp', 'nlp_vector', *ITEM_CATEGORICAL_COLUMNS,
    }
    missing_users = user_required - set(user_frame.columns)
    if missing_users:
        raise ValueError(f'User table is missing: {sorted(missing_users)}')
    item_schema = pl.scan_parquet(item_path).collect_schema()
    missing_items = item_required - set(item_schema.names())
    if missing_items:
        raise ValueError(f'Item table is missing: {sorted(missing_items)}')
    print('User rows:', f'{user_frame.height:,}')
    print('Item schema:', {name: str(item_schema[name]) for name in sorted(item_required)})


def split_eligible_positions(
    video_indices: np.ndarray,
    event_times: np.ndarray,
    upload_times_ms: np.ndarray,
    config: PreprocessConfig,
) -> tuple[np.ndarray, np.ndarray, dict[str, int]]:
    positions = np.arange(config.history_length, len(video_indices), dtype=np.int32)
    if not len(positions):
        empty = np.empty(0, dtype=np.int32)
        return empty, empty.copy(), {
            'excluded_unknown_target': 0,
            'excluded_non_strict_time': 0,
            'excluded_future_upload': 0,
        }

    target_ids = video_indices[positions]
    known_target = target_ids >= 2
    strictly_later = event_times[positions - 1] < event_times[positions]
    target_uploads = upload_times_ms[target_ids]
    upload_available = (target_uploads == MISSING_UPLOAD_MS) | (target_uploads <= event_times[positions])
    eligible = positions[known_target & strictly_later & upload_available]

    count = len(eligible)
    if count == 0:
        train_count = 0
    elif count == 1:
        train_count = 1
    else:
        train_count = min(count - 1, max(1, math.floor(config.train_fraction * count)))

    diagnostics = {
        'excluded_unknown_target': int((~known_target).sum()),
        'excluded_non_strict_time': int((~strictly_later).sum()),
        'excluded_future_upload': int((~upload_available).sum()),
    }
    return eligible[:train_count], eligible[train_count:], diagnostics

## -----------------------
## Config execution test
## -----------------------

CONFIG = PreprocessConfig()
CONFIG.validate()
print('PyTorch:', torch.__version__, '| Polars:', pl.__version__)

if 'df_users' not in globals():
    df_users = pl.read_parquet(USER_PATH)
print('df_users:', df_users.shape)
print('Item Parquet:', ITEM_PATH)


PyTorch: 2.10.0+cu128 | Polars: 1.35.2
df_users: (1000, 26)
Item Parquet: /kaggle/input/datasets/nguyenngocanhle/two-tower-data/KuaiRand-1k-item_table-parquet/kaggle/working/item_table.parquet


The preprocessor writes NumPy memory-mapped arrays under `/kaggle/working/kuairand_retrieval_ready_v2`. For the full KuaiRand-1K catalog, the float32 NLP array is approximately 9 GiB. The function checks free disk space first and writes into a temporary build directory; an interrupted run is not mistaken for a completed cache.

The cache is reused only when its format, configuration, source counts/checksums, and item-file signature match. Use `force_rebuild=True` only when you deliberately want to replace the generated cache.

Category vocabularies are fitted from training information. Unseen category values map to UNK. `target_log_q` represents the actual equal-user training policy:

$$
q(v) = \frac{1}{|U|} \sum_{u \in U} \frac{\text{count of target } v \text{ in user } u\text{'s training pool}}{|T_u|}
$$

It uses no validation target frequencies and no engagement-rate statistics.

In [18]:
def prepare_retrieval_data(
    user_frame: pl.DataFrame,
    item_path: str | Path,
    output_dir: str | Path = OUTPUT_DIR,
    config: PreprocessConfig = CONFIG,
    force_rebuild: bool = False,
) -> dict:
    config.validate()
    item_path = Path(item_path)
    output_dir = Path(output_dir)
    validate_source_schema(user_frame, item_path)

    users = user_frame.select([
        'user_id', 'history_sequence', 'history_time_ms', 'history_feedback',
        'is_lowactive_period', 'user_active_degree', *USER_CATEGORICAL_COLUMNS,
    ]).sort('user_id')
    if not users.height:
        raise ValueError('User table is empty')
    if users['user_id'].null_count() or users['user_id'].n_unique() != users.height:
        raise ValueError('user_id must be non-null and unique')
    if not users['user_id'].dtype.is_integer():
        raise ValueError('This KuaiRand module expects integer user_id values')

    catalog = (
        pl.scan_parquet(item_path)
        .select(['video_id', 'upload_timestamp'])
        .collect()
        .sort('video_id')
    )
    if not catalog.height:
        raise ValueError('Item table is empty')
    if catalog['video_id'].null_count() or catalog['video_id'].n_unique() != catalog.height:
        raise ValueError('video_id must be non-null and unique')
    if not catalog['video_id'].dtype.is_integer():
        raise ValueError('This KuaiRand module expects integer video_id values')

    original_video_ids = catalog['video_id'].to_numpy().astype(np.int64, copy=False)
    if np.any(original_video_ids < 0):
        raise ValueError('video_id must be nonnegative')
    number_of_items = len(original_video_ids) + 2

    upload_values = (
        catalog['upload_timestamp']
        .cast(pl.Float64, strict=False)
        .fill_null(float('nan'))
        .to_numpy()
    )
    finite_upload = np.isfinite(upload_values)
    safe_limit = np.iinfo(np.int64).max // config.upload_seconds_to_ms
    if np.any(np.abs(upload_values[finite_upload]) > safe_limit):
        raise ValueError('upload_timestamp overflows int64 after conversion to milliseconds')
    upload_times_ms = np.full(number_of_items, MISSING_UPLOAD_MS, dtype=np.int64)
    upload_times_ms[np.flatnonzero(finite_upload) + 2] = (
        upload_values[finite_upload].astype(np.int64) * config.upload_seconds_to_ms
    )
    del catalog, upload_values

    user_ids = users['user_id'].to_numpy().astype(np.int64, copy=False)
    source_fingerprint = {
        'user_rows': int(users.height),
        'item_rows': int(len(original_video_ids)),
        'user_id_xor': int(np.bitwise_xor.reduce(user_ids, initial=np.int64(0))),
        'video_id_xor': int(np.bitwise_xor.reduce(original_video_ids, initial=np.int64(0))),
        'total_stored_exposures': int(sum(len(values) if values is not None else 0 for values in users['history_sequence'].to_list())),
        'item_file_bytes': int(item_path.stat().st_size),
        'item_file_mtime_ns': int(item_path.stat().st_mtime_ns),
    }
    config_record = asdict(config)

    if output_dir.exists():
        metadata_path = output_dir / 'metadata.json'
        if force_rebuild:
            shutil.rmtree(output_dir)
        elif metadata_path.is_file():
            metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            if (
                metadata.get('format_version') == FORMAT_VERSION
                and metadata.get('config') == config_record
                and metadata.get('source_fingerprint') == source_fingerprint
                and all((output_dir / name).is_file() for name in metadata.get('artifact_files', []))
            ):
                print('Reusing completed preprocessing:', output_dir)
                return metadata
            raise ValueError(
                f'{output_dir} contains a different or incomplete preprocessing result. '
                'Choose another OUTPUT_DIR or call with force_rebuild=True.'
            )
        else:
            raise ValueError(
                f'{output_dir} exists without metadata.json. '
                'Choose another OUTPUT_DIR or call with force_rebuild=True.'
            )

    positive_original_by_user: list[np.ndarray] = []
    positive_times_by_user: list[np.ndarray] = []
    diagnostics: list[dict] = []
    for row in users.iter_rows(named=True):
        videos = row['history_sequence'] or []
        times = row['history_time_ms'] or []
        feedback = row['history_feedback'] or []
        if not (len(videos) == len(times) == len(feedback)):
            raise ValueError(f"User {row['user_id']}: history fields have different lengths")
        if any(value is None for value in videos) or any(value is None for value in times):
            raise ValueError(f"User {row['user_id']}: history IDs/timestamps contain nulls")

        source_exposure_count = len(videos)
        videos = videos[-config.exposure_limit:]
        times_array = np.asarray(times[-config.exposure_limit:], dtype=np.int64)
        feedback = feedback[-config.exposure_limit:]
        if np.any(times_array[1:] < times_array[:-1]):
            raise ValueError(f"User {row['user_id']}: history is not chronological")

        positive_mask = np.asarray(
            [is_positive_feedback(entry) for entry in feedback], dtype=bool
        )
        positive_original_by_user.append(np.asarray(videos, dtype=np.int64)[positive_mask])
        positive_times_by_user.append(times_array[positive_mask])
        diagnostics.append({
            'user_id': int(row['user_id']),
            'source_exposures': source_exposure_count,
            'retained_exposures': len(videos),
            'positive_events': int(positive_mask.sum()),
        })

    maximum_retained = max(item['retained_exposures'] for item in diagnostics)
    if maximum_retained < config.exposure_limit:
        warnings.warn(
            f'Longest stored history has {maximum_retained} exposures, below the requested '
            f'{config.exposure_limit}. The module cannot recover events absent from user_table.'
        )

    encoded_positive_by_user = [
        encode_video_ids(values, original_video_ids)
        for values in positive_original_by_user
    ]
    event_offsets = np.zeros(users.height + 1, dtype=np.int64)
    event_offsets[1:] = np.cumsum([len(values) for values in encoded_positive_by_user])
    event_video_indices = (
        np.concatenate(encoded_positive_by_user).astype(np.int32, copy=False)
        if event_offsets[-1]
        else np.empty(0, dtype=np.int32)
    )
    event_times_ms = (
        np.concatenate(positive_times_by_user).astype(np.int64, copy=False)
        if event_offsets[-1]
        else np.empty(0, dtype=np.int64)
    )

    train_pairs: list[tuple[int, int]] = []
    validation_pairs: list[tuple[int, int]] = []
    train_pair_offsets = [0]
    training_user_rows: list[int] = []
    training_prefix_video_indices: list[np.ndarray] = []
    target_probabilities = np.zeros(number_of_items, dtype=np.float64)

    for user_row, (videos, times) in enumerate(
        zip(encoded_positive_by_user, positive_times_by_user)
    ):
        train_positions, validation_positions, exclusions = split_eligible_positions(
            videos, times, upload_times_ms, config
        )
        train_pairs.extend((user_row, int(position)) for position in train_positions)
        validation_pairs.extend((user_row, int(position)) for position in validation_positions)
        train_pair_offsets.append(len(train_pairs))

        diagnostics[user_row].update(exclusions)
        diagnostics[user_row].update({
            'unknown_positive_events': int((videos == UNK_INDEX).sum()),
            'training_targets': int(len(train_positions)),
            'validation_targets': int(len(validation_positions)),
        })

        if len(train_positions):
            training_user_rows.append(user_row)
            last_training_position = int(train_positions[-1])
            training_prefix_video_indices.append(videos[:last_training_position + 1])
            np.add.at(
                target_probabilities,
                videos[train_positions],
                1.0 / len(train_positions),
            )
        if len(train_positions) and len(validation_positions):
            if times[train_positions[-1]] >= times[validation_positions[0]]:
                raise AssertionError('Per-user split is not chronological')

    if not training_user_rows:
        raise ValueError('No user has an eligible full training window')

    target_probabilities /= len(training_user_rows)
    if not np.isclose(target_probabilities.sum(), 1.0):
        raise AssertionError('Training target probabilities do not sum to 1')
    target_log_q = np.log(
        np.maximum(target_probabilities, config.log_q_floor)
    ).astype(np.float32)
    del target_probabilities

    train_pairs_array = np.asarray(train_pairs, dtype=np.int32).reshape(-1, 2)
    validation_pairs_array = np.asarray(validation_pairs, dtype=np.int32).reshape(-1, 2)
    training_user_rows_array = np.asarray(training_user_rows, dtype=np.int32)
    if not len(validation_pairs_array):
        warnings.warn('There are no validation targets after preprocessing')

    user_category_vocabs: dict[str, dict[str, int]] = {}
    user_categorical = np.empty(
        (users.height, len(USER_CATEGORICAL_COLUMNS)), dtype=np.int32
    )
    for column_index, column_name in enumerate(USER_CATEGORICAL_COLUMNS):
        all_values = users[column_name].to_list()
        vocab = fit_category_vocab(all_values[row] for row in training_user_rows)
        user_category_vocabs[column_name] = vocab
        user_categorical[:, column_index] = encode_category_values(all_values, vocab)

    user_numeric = np.empty((users.height, 2), dtype=np.float32)
    for row_index, (low_active, active_degree) in enumerate(
        zip(users['is_lowactive_period'], users['user_active_degree'])
    ):
        low_active = 0 if low_active is None else low_active
        if low_active not in (0, 1):
            raise ValueError(f'Unexpected is_lowactive_period value: {low_active!r}')
        user_numeric[row_index] = (float(low_active), map_active_degree(active_degree))

    observed_training_indices = np.unique(
        np.concatenate(training_prefix_video_indices)
    )
    observed_training_indices = observed_training_indices[observed_training_indices >= 2]
    observed_training_original_ids = original_video_ids[observed_training_indices - 2]
    observed_id_series = pl.Series(
        'observed_training_video_ids', observed_training_original_ids
    )
    observed_item_rows = (
        pl.scan_parquet(item_path)
        .select(['video_id', *ITEM_CATEGORICAL_COLUMNS])
        .filter(pl.col('video_id').is_in(observed_id_series))
        .collect()
    )
    item_category_vocabs = {
        name: fit_category_vocab(observed_item_rows[name].to_list())
        for name in ITEM_CATEGORICAL_COLUMNS
    }
    del observed_item_rows, observed_training_original_ids, training_prefix_video_indices

    model_arguments = {
        'video_vocab_size': number_of_items,
        'numeric_dim': 2,
        'user_categorical_cardinalities': [
            len(user_category_vocabs[name]) + 2
            for name in USER_CATEGORICAL_COLUMNS
        ],
        'user_categorical_dims': list(USER_CATEGORICAL_DIMS),
        'item_categorical_cardinalities': [
            len(item_category_vocabs[name]) + 2
            for name in ITEM_CATEGORICAL_COLUMNS
        ],
        'nlp_input_dim': config.nlp_dimension,
    }

    estimated_array_bytes = (
        number_of_items * config.nlp_dimension * np.dtype(np.float32).itemsize
        + number_of_items * len(ITEM_CATEGORICAL_COLUMNS) * np.dtype(np.int32).itemsize
        + number_of_items * (
            np.dtype(np.int64).itemsize + np.dtype(np.float32).itemsize
        )
    )
    required_free_bytes = math.ceil(estimated_array_bytes * 1.05) + 256 * 1024**2
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    available_bytes = shutil.disk_usage(output_dir.parent).free
    print(
        f'Catalog: {number_of_items - 2:,} items | '
        f'item NLP array: {number_of_items * config.nlp_dimension * 4 / 2**30:.2f} GiB'
    )
    if available_bytes < required_free_bytes:
        raise OSError(
            f'Insufficient working disk: need approximately '
            f'{required_free_bytes / 2**30:.2f} GiB; '
            f'{available_bytes / 2**30:.2f} GiB is free'
        )

    stage_dir = Path(tempfile.mkdtemp(
        prefix=f'{output_dir.name}.building-', dir=output_dir.parent
    ))
    item_nlp = None
    item_categorical = None
    try:
        arrays = {
            'original_video_ids': original_video_ids,
            'original_user_ids': user_ids,
            'item_upload_times_ms': upload_times_ms,
            'user_numeric': user_numeric,
            'user_categorical': user_categorical,
            'event_offsets': event_offsets,
            'event_video_indices': event_video_indices,
            'event_times_ms': event_times_ms,
            'train_pairs': train_pairs_array,
            'validation_pairs': validation_pairs_array,
            'train_pair_offsets': np.asarray(train_pair_offsets, dtype=np.int64),
            'training_user_rows': training_user_rows_array,
            'target_log_q': target_log_q,
        }
        for name, array in arrays.items():
            np.save(stage_dir / f'{name}.npy', array, allow_pickle=False)

        item_nlp = np.lib.format.open_memmap(
            stage_dir / 'item_nlp.npy',
            mode='w+', dtype=np.float32,
            shape=(number_of_items, config.nlp_dimension),
        )
        item_categorical = np.lib.format.open_memmap(
            stage_dir / 'item_categorical.npy',
            mode='w+', dtype=np.int32,
            shape=(number_of_items, len(ITEM_CATEGORICAL_COLUMNS)),
        )
        item_nlp[:2] = 0.0
        item_categorical[PAD_INDEX] = PAD_INDEX
        item_categorical[UNK_INDEX] = UNK_INDEX

        written = np.zeros(number_of_items, dtype=bool)
        written[:2] = True
        columns = ['video_id', *ITEM_CATEGORICAL_COLUMNS, 'nlp_vector']
        parquet_file = pq.ParquetFile(item_path)
        processed = 0
        next_progress = 500_000
        for batch in parquet_file.iter_batches(
            batch_size=config.item_batch_size,
            columns=columns,
            use_threads=True,
        ):
            def arrow_column(name: str) -> pa.Array:
                return batch.column(batch.schema.get_field_index(name))

            raw_video_ids = np.asarray(
                arrow_column('video_id').to_numpy(zero_copy_only=False),
                dtype=np.int64,
            )
            item_indices = encode_video_ids(raw_video_ids, original_video_ids)
            if (
                np.any(item_indices < 2)
                or len(np.unique(item_indices)) != len(item_indices)
                or np.any(written[item_indices])
            ):
                raise ValueError('Item table contains missing/duplicate IDs or changed during processing')

            item_nlp[item_indices] = fixed_vectors_to_numpy(
                arrow_column('nlp_vector'), config.nlp_dimension
            )
            for category_index, category_name in enumerate(ITEM_CATEGORICAL_COLUMNS):
                item_categorical[item_indices, category_index] = encode_category_values(
                    arrow_column(category_name).to_pylist(),
                    item_category_vocabs[category_name],
                )
            written[item_indices] = True
            processed += len(item_indices)
            if processed >= next_progress:
                print(f'Encoded item features: {processed:,}/{number_of_items - 2:,}')
                next_progress += 500_000

        if not written.all():
            raise ValueError('Not every catalog item was written to the feature arrays')
        item_nlp.flush()
        item_categorical.flush()
        del item_nlp, item_categorical
        item_nlp = item_categorical = None
        gc.collect()

        pl.DataFrame(diagnostics).write_parquet(
            stage_dir / 'user_split_summary.parquet'
        )
        counts = {
            'users': int(users.height),
            'training_users': int(len(training_user_rows_array)),
            'validation_users': int(
                len(np.unique(validation_pairs_array[:, 0]))
                if len(validation_pairs_array) else 0
            ),
            'catalog_items': int(number_of_items - 2),
            'positive_events': int(len(event_video_indices)),
            'training_target_pool': int(len(train_pairs_array)),
            'validation_examples': int(len(validation_pairs_array)),
            'examples_in_64_pass_epoch': int(len(training_user_rows_array) * 64),
        }
        artifact_files = sorted(path.name for path in stage_dir.iterdir())
        artifact_files.append('metadata.json')
        metadata = {
            'format_version': FORMAT_VERSION,
            'config': config_record,
            'source_fingerprint': source_fingerprint,
            'counts': counts,
            'model_arguments': model_arguments,
            'user_numeric_columns': ['is_lowactive_period', 'user_active_degree'],
            'user_categorical_columns': list(USER_CATEGORICAL_COLUMNS),
            'user_categorical_dims': list(USER_CATEGORICAL_DIMS),
            'item_categorical_columns': list(ITEM_CATEGORICAL_COLUMNS),
            'active_degree_mapping': ACTIVE_DEGREE_TO_CODE,
            'user_category_vocabs': user_category_vocabs,
            'item_category_vocabs': item_category_vocabs,
            'pad_index': PAD_INDEX,
            'unknown_index': UNK_INDEX,
            'split_rule': (
                'Eligible positions are split per user in chronological order: '
                'floor(0.8*N) training and the later remainder validation, with '
                'at least one position in each partition when N>=2; N=1 is training-only.'
            ),
            'target_sampling_probability': (
                'q(v) = mean over training users of count_u(v)/number_of_training_targets_u.'
            ),
            'artifact_files': artifact_files,
            'user_mlp_input_dimension': 150,
        }
        (stage_dir / 'metadata.json').write_text(
            json.dumps(metadata, indent=2), encoding='utf-8'
        )
        os.replace(stage_dir, output_dir)
    except BaseException:
        del item_nlp, item_categorical
        gc.collect()
        shutil.rmtree(stage_dir, ignore_errors=True)
        raise

    print(json.dumps(metadata['counts'], indent=2))
    print('Saved preprocessed data to:', output_dir)
    return metadata


## -------------------------------------
## Training-Validating splitting test
## -------------------------------------

metadata = prepare_retrieval_data(
    user_frame=df_users,
    item_path=ITEM_PATH,
    output_dir=OUTPUT_DIR,
    config=CONFIG,
    force_rebuild=False,
)
user_split_summary = pl.read_parquet(OUTPUT_DIR / 'user_split_summary.parquet')
print(user_split_summary.select([
    pl.len().alias('users'),
    pl.col('positive_events').sum(),
    pl.col('training_targets').sum(),
    pl.col('validation_targets').sum(),
]))
print(user_split_summary.head(10))

User rows: 1,000
Item schema: {'music_type': 'Float64', 'nlp_vector': 'Array(Float32, shape=(512,))', 'tag': 'String', 'upload_timestamp': 'Int64', 'video_id': 'Int32', 'video_type': 'String'}


/tmp/ipykernel_58/1464479633.py:234: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


Catalog: 4,371,868 items | item NLP array: 8.34 GiB
Encoded item features: 507,904/4,371,868
Encoded item features: 1,007,616/4,371,868
Encoded item features: 1,507,328/4,371,868
Encoded item features: 2,007,040/4,371,868
Encoded item features: 2,506,752/4,371,868
Encoded item features: 3,006,464/4,371,868
Encoded item features: 3,506,176/4,371,868
Encoded item features: 4,005,888/4,371,868
{
  "users": 1000,
  "training_users": 934,
  "validation_users": 918,
  "catalog_items": 4371868,
  "positive_events": 71849,
  "training_target_pool": 12139,
  "validation_examples": 3499,
  "examples_in_64_pass_epoch": 59776
}
Saved preprocessed data to: /kaggle/working/kuairand_retrieval_ready_v2
shape: (1, 4)
┌───────┬─────────────────┬──────────────────┬────────────────────┐
│ users ┆ positive_events ┆ training_targets ┆ validation_targets │
│ ---   ┆ ---             ┆ ---              ┆ ---                │
│ u32   ┆ i64             ┆ i64              ┆ i64                │
╞═══════╪═════════

#### Fixed target pools, collator, and DataLoaders

`FixedTargetPool` and `RetrievalBatchCollator` contain no random sampling. The collator receives target positions chosen elsewhere, retrieves each target's immediately preceding 20 positives, and emits keys matching your existing `TwoTowerModel`:

| Key | Shape | Type |
|---|---:|---|
| `history_video_indices` | `(B, 20)` | int64 |
| `history_lengths` | `(B,)` | int64, always 20 |
| `user_numeric` | `(B, 2)` | float32 |
| `user_categorical` | `(B, 18)` | int64 |
| `target_video_indices` | `(B,)` | int64 |
| `item_categorical` | `(B, 3)` | int64 |
| `item_nlp` | `(B, 512)` | float32 |
| `target_log_q` | `(B,)` | float32 |

It also returns `user_row`, `original_user_id`, `target_position`, and `target_time_ms` for diagnostics and evaluation.

In [19]:
class RetrievalStore:
    ARRAY_NAMES = (
        'original_video_ids', 'original_user_ids', 'item_upload_times_ms',
        'user_numeric', 'user_categorical', 'event_offsets',
        'event_video_indices', 'event_times_ms', 'train_pairs',
        'validation_pairs', 'train_pair_offsets', 'training_user_rows',
        'target_log_q', 'item_nlp', 'item_categorical',
    )

    def __init__(self, directory: str | Path = OUTPUT_DIR):
        self.directory = str(Path(directory).resolve())
        root = Path(self.directory)
        self.metadata = json.loads(
            (root / 'metadata.json').read_text(encoding='utf-8')
        )
        if self.metadata['format_version'] != FORMAT_VERSION:
            raise ValueError('Unsupported preprocessing format version')
        self.config = PreprocessConfig(**self.metadata['config'])
        for name in self.ARRAY_NAMES:
            setattr(
                self,
                name,
                np.load(root / f'{name}.npy', mmap_mode='r', allow_pickle=False),
            )

    def __getstate__(self):
        return {'directory': self.directory}

    def __setstate__(self, state):
        self.__init__(state['directory'])

    def decode_video_indices(self, encoded_indices) -> np.ndarray:
        encoded_indices = np.asarray(encoded_indices, dtype=np.int64)
        if (
            np.any(encoded_indices < 2)
            or np.any(encoded_indices >= len(self.original_video_ids) + 2)
        ):
            raise ValueError('Only real video indices in [2, video_vocab_size) can be decoded')
        return np.asarray(self.original_video_ids[encoded_indices - 2])


class FixedTargetPool(Dataset):
    """A deterministic pool of fixed (user row, target position) references."""

    def __init__(self, store: RetrievalStore, split: str):
        if split not in ('train', 'validation'):
            raise ValueError("split must be 'train' or 'validation'")
        self.store = store
        self.split = split

    @property
    def pairs(self) -> np.ndarray:
        return (
            self.store.train_pairs
            if self.split == 'train'
            else self.store.validation_pairs
        )

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> tuple[int, int]:
        if index < 0 or index >= len(self):
            raise IndexError(index)
        user_row, target_position = self.pairs[index]
        return int(user_row), int(target_position)


class RetrievalBatchCollator:
    """Build tensors from targets already selected by the caller."""

    def __init__(self, store: RetrievalStore):
        self.store = store

    def __call__(self, pairs) -> dict[str, torch.Tensor]:
        pairs = np.asarray(pairs, dtype=np.int64).reshape(-1, 2)
        if not len(pairs):
            raise ValueError('Cannot collate an empty batch')

        store = self.store
        user_rows = pairs[:, 0]
        target_positions = pairs[:, 1]
        history_length = store.config.history_length
        if np.any(target_positions < history_length):
            raise AssertionError('A target lacks a full history window')

        flat_targets = store.event_offsets[user_rows] + target_positions
        flat_histories = (
            flat_targets[:, None]
            - history_length
            + np.arange(history_length, dtype=np.int64)[None, :]
        )
        target_video_indices = np.asarray(
            store.event_video_indices[flat_targets], dtype=np.int64
        )
        target_times = np.asarray(store.event_times_ms[flat_targets], dtype=np.int64)
        history_times = np.asarray(store.event_times_ms[flat_histories])

        if np.any(target_video_indices < 2):
            raise AssertionError('PAD/UNK cannot be a target')
        if np.any(history_times >= target_times[:, None]):
            raise AssertionError('Every history event must be strictly earlier than its target')

        def long_tensor(values) -> torch.Tensor:
            return torch.from_numpy(np.array(values, dtype=np.int64, copy=True))

        def float_tensor(values) -> torch.Tensor:
            return torch.from_numpy(np.array(values, dtype=np.float32, copy=True))

        return {
            'user_row': long_tensor(user_rows),
            'original_user_id': long_tensor(store.original_user_ids[user_rows]),
            'target_position': long_tensor(target_positions),
            'target_time_ms': long_tensor(target_times),
            'history_video_indices': long_tensor(
                store.event_video_indices[flat_histories]
            ),
            'history_lengths': torch.full(
                (len(pairs),), history_length, dtype=torch.long
            ),
            'user_numeric': float_tensor(store.user_numeric[user_rows]),
            'user_categorical': long_tensor(store.user_categorical[user_rows]),
            'target_video_indices': long_tensor(target_video_indices),
            'item_categorical': long_tensor(
                store.item_categorical[target_video_indices]
            ),
            'item_nlp': float_tensor(store.item_nlp[target_video_indices]),
            'target_log_q': float_tensor(
                store.target_log_q[target_video_indices]
            ),
        }


def sample_training_batches(
    store: RetrievalStore,
    epoch: int,
    passes_per_epoch: int = 64,
    batch_size: int = 512,
    seed: int = 42,
) -> list[list[int]]:
    """Sample indices into the fixed training pool for one epoch.

    Call this function explicitly inside the training loop. Each pass visits
    every eligible user once. For each user, targets are drawn without
    replacement until that user's pool is exhausted, then the pool is
    reshuffled. Batches never cross pass boundaries, so a user appears at most
    once in a batch.
    """
    if epoch < 0 or seed < 0:
        raise ValueError('epoch and seed must be nonnegative')
    if passes_per_epoch < 1 or batch_size < 1:
        raise ValueError('passes_per_epoch and batch_size must be positive')

    users = np.asarray(store.training_user_rows, dtype=np.int64)
    if not len(users):
        raise ValueError('No users have eligible training targets')
    rng = np.random.default_rng(np.random.SeedSequence([seed, epoch]))

    pools: dict[int, np.ndarray] = {}
    cursors: dict[int, int] = {}
    for user_row in users:
        user_row = int(user_row)
        start, end = map(
            int, store.train_pair_offsets[user_row:user_row + 2]
        )
        if end <= start:
            raise AssertionError('training_user_rows contains an empty pool')
        pools[user_row] = rng.permutation(
            np.arange(start, end, dtype=np.int64)
        )
        cursors[user_row] = 0

    batches: list[list[int]] = []
    for _ in range(passes_per_epoch):
        pass_indices: list[int] = []
        for user_row_value in rng.permutation(users):
            user_row = int(user_row_value)
            if cursors[user_row] == len(pools[user_row]):
                pools[user_row] = rng.permutation(pools[user_row])
                cursors[user_row] = 0
            pass_indices.append(
                int(pools[user_row][cursors[user_row]])
            )
            cursors[user_row] += 1

        batches.extend(
            pass_indices[start:start + batch_size]
            for start in range(0, len(pass_indices), batch_size)
        )
    return batches


def make_training_loader(
    store: RetrievalStore,
    sampled_batches: list[list[int]],
    num_workers: int = 0,
) -> DataLoader:
    """Load an explicit schedule without performing any target sampling."""
    dataset = FixedTargetPool(store, 'train')
    checked_batches: list[list[int]] = []
    for batch in sampled_batches:
        indices = np.asarray(batch)
        if (
            indices.ndim != 1
            or not len(indices)
            or not np.issubdtype(indices.dtype, np.integer)
        ):
            raise ValueError('Each sampled batch must be a nonempty 1-D integer list')
        if np.any(indices < 0) or np.any(indices >= len(dataset)):
            raise IndexError('A sampled index is outside the training target pool')
        user_rows = np.asarray(store.train_pairs[indices, 0])
        if len(np.unique(user_rows)) != len(user_rows):
            raise ValueError('A training batch contains the same user more than once')
        checked_batches.append(indices.astype(np.int64).tolist())

    return DataLoader(
        dataset,
        batch_sampler=checked_batches,
        collate_fn=RetrievalBatchCollator(store),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
    )


def make_validation_loader(
    store: RetrievalStore,
    batch_size: int = 512,
    num_workers: int = 0,
) -> DataLoader:
    """Load every fixed validation example in chronological pool order."""
    return DataLoader(
        FixedTargetPool(store, 'validation'),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=RetrievalBatchCollator(store),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
    )


def print_data_interface(store: RetrievalStore) -> dict:
    arguments = store.metadata['model_arguments']
    print(json.dumps(arguments, indent=2))
    print('User MLP input dimension:', store.metadata['user_mlp_input_dimension'])
    print('Training target pool:', f"{store.metadata['counts']['training_target_pool']:,}")
    print('Fixed validation examples:', f"{store.metadata['counts']['validation_examples']:,}")
    return arguments

## ----------------------------------
## Execute DataLoader
## ----------------------------------

store = RetrievalStore(OUTPUT_DIR)
validation_loader = make_validation_loader(
    store,
    batch_size=512,
    num_workers=0,
)
model_arguments = print_data_interface(store)

{
  "video_vocab_size": 4371870,
  "numeric_dim": 2,
  "user_categorical_cardinalities": [
    4,
    9,
    25,
    378,
    15,
    7,
    5,
    39,
    278,
    9,
    7,
    5,
    4,
    4,
    4,
    4,
    4,
    4
  ],
  "user_categorical_dims": [
    1,
    4,
    8,
    16,
    4,
    8,
    1,
    8,
    16,
    4,
    4,
    4,
    1,
    1,
    1,
    1,
    1,
    1
  ],
  "item_categorical_cardinalities": [
    5,
    7,
    208
  ],
  "nlp_input_dim": 512
}
User MLP input dimension: 150
Training target pool: 12,139
Fixed validation examples: 3,499


In [20]:
def build_two_tower_from_store(store: RetrievalStore, model_config=None):
    required = ('ModelConfig', 'UserTower', 'ItemTower', 'TwoTowerModel')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            'Run the existing two-tower model-definition cell first. '
            f'Missing: {missing}'
        )

    arguments = store.metadata['model_arguments']
    config = ModelConfig() if model_config is None else model_config
    if config.nlp_input_dim != arguments['nlp_input_dim']:
        raise ValueError('Model and preprocessed NLP dimensions differ')

    user_tower = UserTower(
        video_vocab_size=arguments['video_vocab_size'],
        numeric_dim=arguments['numeric_dim'],
        categorical_cardinalities=arguments['user_categorical_cardinalities'],
        categorical_dims=arguments['user_categorical_dims'],
        config=config,
    )
    item_tower = ItemTower(
        video_vocab_size=arguments['video_vocab_size'],
        categorical_cardinalities=arguments['item_categorical_cardinalities'],
        config=config,
    )
    return TwoTowerModel(user_tower, item_tower)

model = build_two_tower_from_store(store)

### 4. Evaluating and Training

#### Evaluation

#### Training

### Export FAISS item